# 3. Despliegue de la API en Cloud9

Este notebook documenta el último tramo del laboratorio: levantar una API que cargue el mejor modelo directamente desde MLflow.

La gracia de este enfoque es que el código del servicio no depende de archivos locales del entrenamiento; en su lugar, depende del `MODEL_URI` almacenado en MLflow.

## Paso 1. Registrar el mejor modelo

Antes de desplegar, conviene registrar la mejor corrida y asignarle el alias `champion`. Así la API puede apuntar siempre a un nombre estable aunque después publiques nuevas versiones.

In [ ]:
!python ../scripts/register_best_model.py --experiment-name imdb-spanish-sentiment --registered-model-name imdb-spanish-sentiment

## Paso 2. Variables de entorno en Cloud9

En la instancia de Cloud9 define, como mínimo:

```bash
export MLFLOW_TRACKING_URI="http://TU_SERVIDOR_MLFLOW:5000"
export MODEL_URI="models:/imdb-spanish-sentiment@champion"
```

La API incluida en `api/main.py` carga el modelo durante el arranque.

In [ ]:
import os

print("MLFLOW_TRACKING_URI =", os.getenv("MLFLOW_TRACKING_URI"))
print("MODEL_URI =", os.getenv("MODEL_URI", "models:/imdb-spanish-sentiment@champion"))

## Paso 3. Levantar el servicio

Desde la terminal de Cloud9 puedes iniciar la API con:

```bash
uvicorn api.main:app --host 0.0.0.0 --port 8080
```

El endpoint principal es `POST /predict` y recibe una lista de reseñas.

In [ ]:
import requests

payload = {
    "texts": [
        "La película fue emocionante y muy bien actuada.",
        "La historia fue lenta, predecible y bastante aburrida."
    ]
}

# Ajusta la URL si la API está expuesta por otro host o puerto.
# response = requests.post("http://localhost:8080/predict", json=payload, timeout=30)
# response.json()

## Checklist de despliegue

- El entorno virtual de Cloud9 tiene instaladas las dependencias del proyecto.
- La máquina puede alcanzar el servidor de MLflow.
- El modelo quedó registrado o al menos existe un `MODEL_URI` válido.
- El puerto elegido está habilitado para pruebas.
- El endpoint `/health` responde correctamente antes de probar `/predict`.

Con esto el laboratorio queda cerrado desde entrenamiento hasta inferencia en una API real.